# Experiment 14A–14F — Combined Research Notebook

This is the synthesis notebook for the **MIN Signal Operator / G_ATLAS Experiment 14 series**. Individual experiment scripts remain the numerical source of truth; this notebook loads their result tables, summarizes the scientific progression, and produces cross-series figures.

## Series map

| Experiment | Scientific question |
|---|---|
| 14A | Does the environment-informed MIN/SOE state contain recoverable symbol information? |
| 14B | How does environment/kernel alignment affect task accessibility? |
| 14C | Does controlled SOE rate geometry alter task-relevant state structure? |
| 14D | How much of the geometry effect is explained by readout conditioning? |
| 14E | Does causal memory help predict a future observation in the IID-symbol setting? |
| 14F | Does causal MIN/SOE memory expose predictive information in a genuinely hidden-memory process? |

The intended interpretation is a chain of increasingly specific tests, not a single MIN performance score.


## 14A — Task-relevant subspace

14A establishes the first task-level bridge: digital symbols → environment-corrupted observation → environment-informed MIN/SOE state → PCA → linear readout.

The corrected raw control uses the **noisy observation** at the symbol center. The experiment measures held-out symbol NMSE versus retained complex PCA dimension and defines an operational task dimension.

Boundary: 14A establishes recoverability and task-relevant structure under a declared decoder; it does not establish that MIN is superior to raw observations.


## 14B — Environment/kernel alignment

14B compares informed, estimated, mismatched and raw-center representations. The purpose is to test whether task accessibility depends on the relationship between the represented memory kernel and the actual environment.

Alignment effects are conditional on the same signal, environment, SNR, seed and readout protocol; they are not a universal kernel-identification claim.


## 14C — Geometry → task relevance

14C holds the observed waveform fixed within paired cases and varies only the 16-mode SOE rate geometry: clustered, logspread and wide.

This isolates temporal-rate geometry from waveform variation. The result establishes that nominal mode count does not determine task accessibility. It is not a geometry leaderboard.


## 14D — Readout accessibility and conditioning

14D asks whether the 14C geometry effect is merely decoder pathology. It uses stable SVD least squares for the unregularized case and a declared ridge grid.

The authoritative result showed severe conditioning for clustered states, while logspread and wide were numerically full rank. Regularization changed the magnitude of differences but did not remove them.

Therefore decoder conditioning explains part, but not all, of the observed geometry dependence.


## 14E — Causal prediction in the IID-symbol setting

Because the communication symbols are IID, next-symbol prediction would not be a legitimate memory test. 14E instead predicts the next observed waveform sample at the following symbol center using only information through the current center.

This is deliberately a constraint experiment. The simple current-observation baseline already captures much of the available predictive structure; MIN therefore does not receive a generic predictive-superiority claim from the IID setting.


## 14F — Hidden-memory Markovization

14F changes the process itself. Sixteen latent exponentially correlated modes form a Markov hidden vector; their scalar mixture is observed with independent measurement noise. Predictors compare the current observation, explicit finite histories, causal MIN/SOE state, and the true hidden state oracle.

This is the direct controlled test of the stronger temporal-state postulate: whether causal memory augmentation can expose predictive information unavailable from the instantaneous observation.


## Dimension discipline

The series keeps distinct:

$L \neq D_{eff} \neq D_{basis} \neq D_{state} \neq D_{task}$.

Task dimension is an operational readout quantity. Predictive improvement is not equivalent to hidden-state recovery, and neither is evidence of a universal MIN advantage.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "experiments").exists() and (ROOT.parent / "experiments").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "experiments" / "results"

FILES = {
    "14A": "14A_task_relevant_results.csv",
    "14B": "14B_alignment_results.csv",
    "14C": "14C_geometry_to_task_results.csv",
    "14D": "14D_readout_stability_results.csv",
    "14E": "14E_causal_prediction_results.csv",
    "14F": "14F_hidden_memory_markovization_results.csv",
}

tables = {}
for name, filename in FILES.items():
    path = RESULTS / filename
    if path.exists():
        tables[name] = pd.read_csv(path)
        print(f"{name}: {len(tables[name]):,} rows")
    else:
        print(f"{name}: result CSV not present locally")


In [ ]:
## 14C — geometry curves

if "14C" in tables:
    df = tables["14C"]
    metric = "heldout_symbol_nmse"
    if metric in df.columns:
        g = df[df["representation"].eq("full_state")]
        g = g[g["retained_dimension"].isin([1,2,4,8,12,16])]
        s = g.groupby(["geometry","retained_dimension"])[metric].mean().reset_index()
        for geometry in s["geometry"].unique():
            z = s[s["geometry"].eq(geometry)]
            plt.plot(z["retained_dimension"], z[metric], marker="o", label=geometry)
        plt.xlabel("Retained PCA dimension")
        plt.ylabel("Mean held-out NMSE")
        plt.title("14C — SOE geometry and task accessibility")
        plt.legend()
        plt.show()


In [ ]:
## 14D — regularization

if "14D" in tables and "heldout_symbol_nmse" in tables["14D"].columns:
    df = tables["14D"]
    g = df[(df["representation"].eq("full_state")) & (df["retained_dimension"].eq(16))]
    s = g.groupby(["geometry","ridge_lambda"])["heldout_symbol_nmse"].mean().reset_index()
    for geometry in s["geometry"].unique():
        z = s[s["geometry"].eq(geometry) & s["ridge_lambda"].gt(0)]
        plt.semilogx(z["ridge_lambda"], z["heldout_symbol_nmse"], marker="o", label=geometry)
    plt.xlabel("Ridge lambda")
    plt.ylabel("Mean held-out NMSE")
    plt.title("14D — readout regularization sensitivity")
    plt.legend()
    plt.show()


In [ ]:
## 14E — causal prediction

if "14E" in tables:
    df = tables["14E"]
    g = df[(df["representation"].eq("full_state")) & (df["retained_dimension"].eq(16)) & (df["ridge_lambda"].eq(1e-6))]
    if not g.empty:
        print(g.groupby("geometry")["heldout_next_observation_nmse"].agg(["mean","median","std"]))


In [ ]:
## 14F — hidden-memory predictive hierarchy

if "14F" in tables:
    df = tables["14F"]
    m = df[(df["representation"].eq("full_state")) & (df["retained_dimension"].eq(16)) & (df["ridge_lambda"].eq(1e-6))]
    best_min = m.groupby(["source_geometry","snr_db","seed"])["heldout_next_state_nmse"].min().mean()
    h = df[df["representation"].eq("finite_history")].groupby(["source_geometry","snr_db","seed"])["heldout_next_state_nmse"].min().mean()
    o = df[df["representation"].eq("oracle_hidden_state")]["heldout_next_state_nmse"].mean()
    raw = df.groupby(["source_geometry","snr_db","seed"])["current_observation_nmse"].first().mean()
    print({"current_observation":raw, "best_MIN_d16":best_min, "best_finite_history":h, "oracle_hidden_state":o})


## Cross-series conclusion

The completed series supports a deliberately narrow chain:

1. 14A: task information is recoverable from the constructed MIN/SOE state.
2. 14B: representation/environment alignment can alter accessibility.
3. 14C: temporal rate geometry changes task-relevant state structure.
4. 14D: decoder conditioning contributes, but does not exhaust, that effect.
5. 14E: an IID communication source does not justify a generic predictive-memory claim.
6. 14F: when genuine hidden temporal dependence is introduced, causal memory access can recover predictive information beyond the instantaneous observation.

The strongest defensible statement is therefore that **finite memory-state representations can expose task-relevant and predictive information whose accessibility depends on the task, temporal geometry, observation quality and readout**.

## Reproducibility boundary

This notebook is a synthesis layer. Exact numerical reproduction belongs to the individual experiment scripts and authoritative CI artifacts. Missing local CSVs are reported rather than silently substituted.

Do not infer a universal MIN advantage, a best SOE geometry, exact hidden-state recovery, or generalization beyond the controlled processes tested.
